# R2 — Silver rebuild + quarantine

Rebuilds one row per order from the bronze event stream, deduplicating events and quarantining broken orders. The transformation code runs in the **hop_pipeline** Spark Declarative Pipeline (`src/hop_pipeline/transformations/order_events_dedup.py` and `orders_rebuild.py`); this notebook documents the round and verifies its outputs.

In [0]:
# Config cell — change model names here only, if a built-in model name changes
CATALOG = "hop"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"
print(CATALOG, CHAT_MODEL, EMBED_MODEL)

## What R2 builds (in the hop_pipeline SDP)

* `hop.silver.order_events_dedup` — one row per `event_id` (earliest `_ingest_ts` wins)
* `hop.silver.orders` — one row per `order_id`: earliest `event_ts` per event type; `rider_id` from picked_up (or delivered); `cancelled_by` from cancelled; status delivered/cancelled
* `hop.silver.orders_quarantine` — orders breaking any rule, with `failed_rules`: Q-A unknown references, Q-B terminal-event count != 1, Q-C timestamps out of order, Q-D bad value/class/promise

In [0]:
# R2 — verification (read-only): counts, rule mix, status split
from pyspark.sql import functions as F

orders = spark.table("hop.silver.orders")
quar = spark.table("hop.silver.orders_quarantine")
dedup = spark.table("hop.silver.order_events_dedup")

n_orders, n_quar = orders.count(), quar.count()
print("dedup rows:", dedup.count(), "| orders:", n_orders, "| quarantine:", n_quar,
      "| total:", n_orders + n_quar, "(expect 13195)")

quar.groupBy("failed_rules").count().orderBy(F.col("count").desc()).show(truncate=False)
orders.groupBy("status").count().show()
display(orders.limit(5))